In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count
from pyspark.ml.feature import StringIndexer
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.feature import StandardScaler
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Extract

In [0]:
spark = SparkSession.builder \
    .appName("Assignment_3_ETL") \
    .getOrCreate()

In [0]:
df = spark.read.csv(
    "/Volumes/workspace/default/assignment_data/adult.csv",
    header=True,
    inferSchema=True
)

display(df.limit(5))
df.printSchema()
print("Rows:", df.count())
print("Columns:", len(df.columns))

age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K


root
 |-- age: integer (nullable = true)
 |-- workclass: string (nullable = true)
 |-- fnlwgt: integer (nullable = true)
 |-- education: string (nullable = true)
 |-- education.num: integer (nullable = true)
 |-- marital.status: string (nullable = true)
 |-- occupation: string (nullable = true)
 |-- relationship: string (nullable = true)
 |-- race: string (nullable = true)
 |-- sex: string (nullable = true)
 |-- capital.gain: integer (nullable = true)
 |-- capital.loss: integer (nullable = true)
 |-- hours.per.week: integer (nullable = true)
 |-- native.country: string (nullable = true)
 |-- income: string (nullable = true)

Rows: 32561
Columns: 15


# Transform

In [0]:
df = df.toDF(*[
    c.replace(".", "_")
     .replace("-", "_")
     .replace(" ", "_")
    for c in df.columns
])

In [0]:
# Handling missing values

display(
    df.select(
        count(when(col("workclass")=="?",True)).alias("workclass"),
        count(when(col("occupation")=="?",True)).alias("occupation"),
        count(when(col("native_country")=="?",True)).alias("native_country")
    )
)

workclass,occupation,native_country
1836,1843,583


In [0]:
# Removing missing values and duplicates

df = df.filter(col("workclass") != "?")
df = df.filter(col("occupation") != "?")
df = df.filter(col("native_country") != "?")

df = df.dropDuplicates()

## Feature Engineering

In [0]:
categorical_cols = [
    "workclass",
    "education",
    "marital_status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "native_country"
]

for column in categorical_cols:

    indexer = StringIndexer(
        inputCol=column,
        outputCol=column+"_index",
        handleInvalid="keep"
    )

    df = indexer.fit(df).transform(df)

## VectorAssembler

In [0]:
feature_columns = [
    "age",
    "fnlwgt",
    "education_num",
    "capital_gain",
    "capital_loss",
    "hours_per_week",
    "workclass_index",
    "education_index",
    "marital_status_index",
    "occupation_index",
    "relationship_index",
    "race_index",
    "sex_index",
    "native_country_index"
]

assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features"
)

df = assembler.transform(df)

In [0]:
scaler = StandardScaler(
    inputCol="features",
    outputCol="scaled_features"
)

scaler_model = scaler.fit(df)

df = scaler_model.transform(df)

# Load

In [0]:
df.write.mode("overwrite").parquet(
    "/Volumes/workspace/default/assignment_data/processed_data.parquet"
)

In [0]:
# load processed data

processed_df = spark.read.parquet(
    "/Volumes/workspace/default/assignment_data/processed_data.parquet"
)

In [0]:
# verify it
display(processed_df.limit(5))
print("Rows:", processed_df.count())
print("Columns:", len(processed_df.columns))

age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income,workclass_index,education_index,marital_status_index,occupation_index,relationship_index,race_index,sex_index,native_country_index,features,scaled_features
34,Private,203034,Bachelors,13,Separated,Sales,Not-in-family,White,Male,0,2824,50,United-States,>50K,0.0,2.0,3.0,4.0,1.0,0.0,0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""34.0"",""203034.0"",""13.0"",""0.0"",""2824.0"",""50.0"",""0.0"",""2.0"",""3.0"",""4.0"",""1.0"",""0.0"",""0.0"",""0.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""2.589208519643689"",""1.9216036671611085"",""5.100562588787359"",""0.0"",""6.982403869162832"",""4.1740571348126725"",""0.0"",""0.6328610656755982"",""2.767694777640461"",""1.3499447036258276"",""0.7378004508775404"",""0.0"",""0.0"",""0.0""]}"
59,Self-emp-inc,107287,10th,6,Widowed,Exec-managerial,Unmarried,White,Female,0,2559,50,United-States,>50K,4.0,7.0,4.0,2.0,3.0,0.0,1.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""59.0"",""107287.0"",""6.0"",""0.0"",""2559.0"",""50.0"",""4.0"",""7.0"",""4.0"",""2.0"",""3.0"",""0.0"",""1.0"",""0.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""4.493038313499342"",""1.0154116681871699"",""2.3541058102095502"",""0.0"",""6.327185375774676"",""4.1740571348126725"",""3.0797274219575552"",""2.2150137298645936"",""3.6902597035206144"",""0.6749723518129138"",""2.213401352632621"",""0.0"",""2.136263471386716"",""0.0""]}"
51,State-gov,68898,Assoc-voc,11,Divorced,Tech-support,Not-in-family,White,Male,0,2444,39,United-States,>50K,3.0,4.0,2.0,10.0,1.0,0.0,0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""51.0"",""68898.0"",""11.0"",""0.0"",""2444.0"",""39.0"",""3.0"",""4.0"",""2.0"",""10.0"",""1.0"",""0.0"",""0.0"",""0.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""3.8838127794655337"",""0.6520811758624963"",""4.3158606520508425"",""0.0"",""6.042845274870382"",""3.2557645651538847"",""2.3097955664681664"",""1.2657221313511964"",""1.8451298517603072"",""3.374861759064569"",""0.7378004508775404"",""0.0"",""0.0"",""0.0""]}"
55,Self-emp-inc,124137,Prof-school,15,Married-civ-spouse,Prof-specialty,Husband,White,Male,0,2415,35,Greece,>50K,4.0,9.0,0.0,0.0,0.0,0.0,0.0,26.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""55.0"",""124137.0"",""15.0"",""0.0"",""2415.0"",""35.0"",""4.0"",""9.0"",""0.0"",""0.0"",""0.0"",""0.0"",""0.0"",""26.0""]}","{""type"":""1"",""size"":null,""indices"":null,""values"":[""4.1884255464824385"",""1.1748875283468707"",""5.885264525523876"",""0.0"",""5.971142118990169"",""2.9218399943688707"",""3.0797274219575552"",""2.847874795540192"",""0.0"",""0.0"",""0.0"",""0.0"",""0.0"",""6.297737361532431""]}"
41,Local-gov,297248,Prof-school,15,Married-civ-spouse,Prof-specialty,Husband,White,Male,0,2415,45,United-States,>50K,2.0,9.0,0.0,0.0,0.0,0.0,0.0,0.0,"{""type"":""0"",""size"":""14"",""indices"":[""0"",""1"",""2"",""4"",""5"",""6"",""7""],""values"":[""41.0"",""297248.0"",""15.0"",""2415.0"",""45.0"",""2.0"",""9.0""]}","{""type"":""0"",""size"":""14"",""indices"":[""0"",""1"",""2"",""4"",""5"",""6"",""7""],""values"":[""3.1222808619232723"",""2.813286675415473"",""5.885264525523876"",""5.971142118990169"",""3.7566514213314055"",""1.5398637109787776"",""2.847874795540192""]}"


Rows: 30139
Columns: 25


In [0]:
# Encode Target Variable

label_indexer = StringIndexer(
    inputCol="income",
    outputCol="label"
)

processed_df = label_indexer.fit(processed_df).transform(processed_df)

In [0]:
# Train test spilit

train, test = processed_df.randomSplit([0.8, 0.2], seed=42)

In [0]:
# Decision Tree Model

dt = DecisionTreeClassifier(
    featuresCol="scaled_features",
    labelCol="label"
)

model = dt.fit(train)

In [0]:
# Prediction

predictions = model.transform(test)

In [0]:
# ACCURACY 

evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

accuracy = evaluator.evaluate(predictions)

print("Accuracy:", accuracy)

Accuracy: 0.8418069678279714


In [0]:
# Display Prediction 

display(
    predictions.select(
        "label",
        "prediction",
        "probability"
    ).limit(10)
)

print(model.featureImportances)

label,prediction,probability
0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9469989746825459"",""0.053001025317454056""]}"
0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9469989746825459"",""0.053001025317454056""]}"
0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9469989746825459"",""0.053001025317454056""]}"
0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9469989746825459"",""0.053001025317454056""]}"
0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9469989746825459"",""0.053001025317454056""]}"
0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9469989746825459"",""0.053001025317454056""]}"
0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9469989746825459"",""0.053001025317454056""]}"
0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9469989746825459"",""0.053001025317454056""]}"
0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9469989746825459"",""0.053001025317454056""]}"
0.0,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.9469989746825459"",""0.053001025317454056""]}"


(14,[0,2,3,4,5,8,9],[0.0019311870081539265,0.2163090995887335,0.2179230393820615,0.01585482456544383,0.007883066455817036,0.5396410188907919,0.000457764108998299])
